# 📌 Prompt Caching

![Topic](https://img.shields.io/badge/Topic-Prompt_Caching-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-June%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — Prompt caching is an advanced optimization technique for Large Language Models that strategically stores and reuses the heavy computational work performed on repeated or static content. **It cuts both latency and API costs, sometimes dramatically (up to 60–90% on cached tokens.)**</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install google-genai` |


---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

In practical scenarios, users often ask multiple questions about the same document or engage in multi-turn conversations with persistent context. However, every time you send a request to an LLM, the model has to read and process every single token in your prompt before it can even start generating a reply. And a long system prompt increases time-to-first-token (TTFT) because the model processes every token through its attention mechanism before producing any output, that "prefill" computation is expensive, and it runs on every single request. 

Without optimization, the model reprocesses the same context repeatedly, resulting in unnecessary expenses and slower responses. Imagine a research paper analysis tool: a researcher uploads a 40-page scientific paper of approximately 30,000 tokens and asks ten different questions. If the model has to re-read and re-analyze all 30,000 tokens for each question, you're paying for 300,000 tokens — when, in reality, the paper itself hasn't changed. 
Prompt caching solves this by letting the model "remember" the work it already did on that shared, static content.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

Prompt caching stores the computational state from an LLM's attention layers so the model can skip redundant prefill work on repeated prompt prefixes. More technically, it stores the key-value (KV) states of attention layers for previously processed tokens, avoiding repeated computation of these cached KV states when generating the next token. Redisarxiv

### 2.1 Prefix-based caching

Reuses the KV vectors of repeated tokens in previous requests, speeding up the prefill stage and reducing TTFT. This is the most common approach and requires the cached content to appear at the start of the prompt.

### 2.2 Explicit/developer-controlled caching 

Anthropic provides developer-controlled caching through explicit cache breakpoints, allowing users to specify which portions of their prompt should be cached, with configurable time-to-live (TTL) options. 


### 2.3 Implicit/automatic caching

Implicit caching is enabled by default for all Gemini 2.5 and newer models. Cost savings are automatically passed on if your request hits caches — there is nothing you need to do in order to enable this.

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Cost reduction** | Implicit caching provides a 90% discount on cached tokens compared to standard input tokens |
| 🟢 | **Latency improvement** | Prompt caching dramatically reduces both costs and response times by eliminating redundant prefill on every request, lowering TTFT. |
| 🟢 | **Use-case versatility** | LLM agents can cache complex tool definitions and instructions for multi-step automation, making it useful for chatbots, RAG, and document Q&A alike. |
| 🟢 | **Provider flexibility** | Major LLM providers have implemented prompt caching with varying approaches — OpenAI offers automatic caching, Anthropic provides developer-controlled breakpoints, and Google offers both implicit and explicit caching. |
| 🔴 | **Prefix constraint** | The main constraint is prefix matching — prompt caching works by comparing the beginning of your current prompt against what's already cached. Changing content early in the prompt breaks the cache. |
| 🔴 | **Minimum token thresholds** | Minimum token thresholds (typically 1,024–4,096 tokens depending on the model) mean caching only activates for sufficiently long prompts. Short prompts get no benefit. |
| 🔴 | **TTL expiry** | TTL durations range from 5 minutes to 24 hours depending on provider and configuration — caches expire and must be rebuilt, so very infrequent traffic may not benefit. |
| 🔴 | **Storage costs (explicit)** | For explicit caching, there are also storage costs based on how long caches are stored, though there are no storage costs for implicit caching.|

---
## 4. Code Example


In [ ]:
import os
from google import genai
from google.genai.types import CreateCachedContentConfig, GenerateContentConfig

# 1. Initialize the Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Load your large, reusable document (e.g. a knowledge base)
#with open("knowledge_base.txt", "r") as f:
    #large_document = f.read()
large_document = "This is a large document that contains important information. " * 1000  # Simulating a large document by repeating a string

# 3. Create the cache — this is the one-time "upload" step
#    The model reads and stores the KV state for this content.
cache = client.caches.create(
    model="gemini-1.5-flash-001",  # Must use a versioned model
    config=CreateCachedContentConfig(
        display_name="My Knowledge Base Cache",
        system_instruction=(
            "You are a helpful assistant. "
            "Answer questions using the provided document."
        ),
        contents=[large_document],
        ttl="900s",  # Cache lives for 15 minutes
    ),
)

print(f"Cache created: {cache.name}")
print(f"Tokens cached: {cache.usage_metadata.total_token_count}")
print(f"Expires at:    {cache.expire_time}")

# 4. Send multiple queries — each one REUSES the cached content.
#    The large document is NOT re-sent or re-processed each time.
queries = [
    "What are the main themes in the document?",
    "Summarize section 3.",
    "What conclusions does the document draw?",
]

for query in queries:
    response = client.models.generate_content(
        model="gemini-1.5-flash-001",
        contents=query,
        config=GenerateContentConfig(cached_content=cache.name),
    )
    
    # Inspect token savings
    meta = response.usage_metadata
    print(f"\nQuery: {query}")
    print(f"  Total tokens:  {meta.total_token_count}")
    print(f"  Cached tokens: {meta.cached_content_token_count}")  # ← the savings
    print(f"  Output tokens: {meta.candidates_token_count}")
    print(f"  Answer: {response.text[:100]}...")

# 5. Clean up the cache when done
client.caches.delete(name=cache.name)
print("\nCache deleted.")

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **Prompt caching stores the KV attention state, not just the text.**
- **Your static content must come first.**
- **Different providers, different mechanics.**
- **The gains compound at scale.**
- **Cache hits are measurable.**

</div>